In [ ]:
import requests
import pandas as pd
from io import StringIO
import sqlite3
import time
import re
from Bio import Entrez

**Data Selection & Extraction**

**UniProt**

Below we are targeting the Human Proteome to provide a structured foundation for mapping drug activities and protein structures later in the pipeline.

Dataset 1: UniProtKB (Swiss-Prot)
Source: UniProt REST API.
Selection Criteria:
* Organism: Homo sapiens (Taxonomy ID: 9606).
* Quality: Reviewed (Swiss-Prot) entries only, ensuring high-quality, manually curated data.
* Key Attributes Extracted:
    * `accession`: The unique primary identifier (common join key).
    * `gene_names`: Standard genetic nomenclature for user queries.
    * `protein_name`: Descriptive biological name of the target.
* Format: Tab-Separated Values (TSV).

Extraction logic: We utilize the UniProt `/stream` endpoint to bypass standard pagination limits, allowing us to retrieve all ~20,000 human reviewed proteins in a single request. The data is read into a pandas DataFrame using StringIO to simulate a file-like object for efficient loading.

In [ ]:
def fetch_human_swissprot():
    print("Fetching data from UniProt... (this may take a few seconds)")


    url = "https://rest.uniprot.org/uniprotkb/stream"

    # Parameters:
    # organism_id:9606 is Human
    # reviewed:true is Swiss-Prot

    # Perhaps include here the filter by proteins that have 3d structures available
    params = {
        "query": "organism_id:9606 AND reviewed:true",
        #"query": "(organism_id:9606) AND (reviewed:true) AND (structure_3d:true)",
        "fields": "accession,gene_names,protein_name",
        "format": "tsv"
    }

    response = requests.get(url, params=params)

    if response.ok:

        data = StringIO(response.text)
        df = pd.read_csv(data, sep='\t')

        print(f"Success! Retrieved {len(df)} reviewed human proteins.")
        return df
    else:
        print(f"Failed to retrieve data. Status code: {response.status_code}")
        return None


df_proteins = fetch_human_swissprot()
df_proteins.to_csv("human_swissprot_uniprot.csv", index=False)
df_proteins.head(10)

Fetching data from UniProt... (this may take a few seconds)
Success! Retrieved 20431 reviewed human proteins.


,Entry,Gene Names,Protein names
0,A0A087X1C5,CYP2D7,Cytochrome P450 2D7 (EC 1.14.14.1)
1,A0A096LP01,SMIM26 LINC00493,Small integral membrane protein 26
2,A0A0B4J2F0,PIGBOS1,Protein PIGBOS1 (PIGB opposite strand protein 1)
3,A0A0C5B5G6,MT-RNR1,Mitochondrial-derived peptide MOTS-c (Mitochon...
4,A0A0K2S4Q6,CD300H,Protein CD300H (CD300 antigen-like family memb...
5,A0A0U1RRE5,NBDY LINC01420,Negative regulator of P-body association (P-bo...
6,A0A1B0GTW7,CIROP LMLN2,Ciliated left-right organizer metallopeptidase...
7,A0A2R8Y7D0,TINCR LINC00036 NCRNA00036 PLAC2,Ubiquitin domain-containing protein TINCR (Pla...
8,A0A8I5KQE6,RPSA2 RPSA RPSAP58,Small ribosomal subunit protein uS2B (37 kDa l...
9,A0AV02,SLC12A8 CCC9,Solute carrier family 12 member 8 (Cation-chlo...


**ChEMBL database**

This phase focuses on enriching our protein list with experimental bioactivity data. We integrate the structured UniProt identifiers with the ChEMBL 36 database, a local repository of bioactive molecules and their targets downloaded from https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/latest/

Dataset 2: ChEMBL Bioactivity Database
* Source: Local SQLite database (chembl_36.db).
* Relationship: Linked to the UniProt dataset via the accession (UniProt ID) identifier.
* Key Attributes Extracted:
    * Chemical: Drug ChEMBL IDs and preferred names.
    * Biological: Target names and bioactivity metrics (IC50, Ki, Kd).
    * Bibliographic: PubMed IDs, article titles, and journal names for textual search.

To resolve the challenge of filtering a massive database (ChEMBL) against a large local list (20,431 proteins), we implemented a Temporary Table Strategy:
* Normalization: Unique UniProt accessions are loaded into a temporary SQL table (my_protein_filter).
* Schema Alignment: We perform a multi-table JOIN across 7 tables (activities, molecule_dictionary, target_dictionary, etc.) to align chemical results with biological sequences.
* Data Cleaning: We filter for SINGLE PROTEIN target types to ensure data granularity and exclude records with null standard_values to ensure data quality.

Integration Summary:
* Input: 20,431 UniProt IDs.
* Filter Logic: Strict match on Single Human Proteins with valid numerical experimental results.
* Result: A refined dataset of proteins that have both a curated biological identity and a published pharmacological profile.

In [ ]:
#path tho chemble_36.db on local PC
db_path = r"C:\Users\Anna_Maksymchuk1\Desktop\Andre\Chembl\chembl_36\chembl_36_sqlite\chembl_36.db"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

try:

    cursor.execute("CREATE TEMP TABLE my_protein_filter (uniprot_id TEXT)")

    protein_ids = [(pid,) for pid in df_proteins['Entry'].unique()]
    cursor.executemany("INSERT INTO my_protein_filter VALUES (?)", protein_ids)
    print(f"Filter table ready with {len(protein_ids)} human proteins.")

    query = """
    SELECT
        act.activity_id,
        mol.chembl_id AS drug_chembl_id,
        mol.pref_name AS drug_name,
        tar.chembl_id AS target_chembl_id,
        tar.pref_name AS target_name,
        tar.organism,
        act.standard_type,
        act.standard_value,
        act.standard_units,
        act.pchembl_value,
        ass.assay_type,
        ass.description AS assay_description,
        ass.assay_organism,
        ass.confidence_score,
        doc.title AS article_title,
        doc.journal,
        doc.year,
        doc.pubmed_id,
        seq.accession AS uniprot_id
    FROM activities act
    JOIN molecule_dictionary mol ON act.molregno = mol.molregno
    JOIN assays ass ON act.assay_id = ass.assay_id
    JOIN target_dictionary tar ON ass.tid = tar.tid
    JOIN docs doc ON ass.doc_id = doc.doc_id
    JOIN target_components tc ON tar.tid = tc.tid
    JOIN component_sequences seq ON tc.component_id = seq.component_id
    -- JOIN against our temporary list to filter by your Swiss-Prot IDs
    JOIN my_protein_filter filter ON seq.accession = filter.uniprot_id
    WHERE tar.target_type = 'SINGLE PROTEIN'
    AND act.standard_value IS NOT NULL
    """


    df_results = pd.read_sql_query(query, conn)

    print(f"Success! Found {len(df_results)} activities for specific proteins.")

    df_results.to_csv("human_swiss_prot_activities.csv", index=False)
    display(df_results.head())

except Exception as e:
    print(f"Error occurred: {e}")

finally:
    conn.close()

Filter table ready with 20431 human proteins.
Success! Found 6441095 activities for specific proteins.


,activity_id,drug_chembl_id,drug_name,target_chembl_id,target_name,organism,standard_type,standard_value,standard_units,pchembl_value,assay_type,assay_description,assay_organism,confidence_score,article_title,journal,year,pubmed_id,uniprot_id
0,1132478,CHEMBL280487,OKADAIC ACID,CHEMBL4438,Serine/threonine-protein phosphatase PP1-gamma...,Homo sapiens,IC50,20.0,nM,7.70,B,Inhibitory activity against serine/threonine p...,None,8,Self-association of okadaic acid upon complexa...,J Med Chem,2004.0,14695814.0,P36873
1,830862,CHEMBL177285,None,CHEMBL1790,Vasopressin V2 receptor,Homo sapiens,IC50,50.0,nM,7.30,B,Inhibitory activity against human recombinant ...,None,8,Synthesis and structure-activity relationships...,J Med Chem,2004.0,14695824.0,P30518
2,831930,CHEMBL175182,None,CHEMBL1790,Vasopressin V2 receptor,Homo sapiens,IC50,60.0,nM,7.22,B,Inhibitory activity against human recombinant ...,None,8,Synthesis and structure-activity relationships...,J Med Chem,2004.0,14695824.0,P30518
3,831936,CHEMBL177885,None,CHEMBL1790,Vasopressin V2 receptor,Homo sapiens,IC50,8.0,nM,8.10,B,Inhibitory activity against human recombinant ...,None,8,Synthesis and structure-activity relationships...,J Med Chem,2004.0,14695824.0,P30518
4,834214,CHEMBL366816,None,CHEMBL1790,Vasopressin V2 receptor,Homo sapiens,IC50,10000.0,nM,NaN,B,Inhibitory activity against human recombinant ...,None,8,Synthesis and structure-activity relationships...,J Med Chem,2004.0,14695824.0,P30518


**PDBe Graph API**

In this step, we connect protein sequences and their physical 3D structures.

Dataset 3: Protein Data Bank Europe (PDBe)

* Source: PDBe Graph API (mappings/best_structures).
* Relationship: Linked to the previous datasets via the uniprot_id.
* Key Attributes Extracted:
    * pdb_id: The 4-character unique identifier for the 3D structure.
    * resolution: A numerical quality metric (lower is better) indicating the clarity of the 3D model.
    * coverage: The percentage of the protein sequence that is actually visible in the 3D structure.
    * unp_start / unp_end: The specific amino acid coordinates mapped to the structure.


Because many human proteins have hundreds of associated structures, we utilize the /best_structures/ endpoint. This API automatically filters out low-quality entries, providing a curated list of high-resolution models.

Key Technical Implementations:
* Rate Limiting: A time.sleep(0.1) delay is included to comply with EBI server requirements and prevent API blocking.
* Handling One-to-Many Relationships: A single UniProt ID often maps to multiple PDB IDs. The logic stores these as separate rows in df_pdb_info, creating a structural library.
* Data Cleaning: Regular expressions are used to normalize uniprot_id strings, ensuring no whitespace or special characters interfere with the API request.


In [ ]:
df_results = pd.read_csv("human_swiss_prot_activities.csv")
df_results['uniprot_id'] = df_results['uniprot_id'].astype(str).apply(lambda x: re.sub(r'[^a-zA-Z0-9]', '', x))
unique_uids = df_results['uniprot_id'].unique()

pdb_data = []

print(f"Starting PDB fetch for {len(unique_uids)} unique proteins...")

for i, uid in enumerate(unique_uids):
    try:
        url = f"https://www.ebi.ac.uk/pdbe/graph-api/mappings/best_structures/{uid}"
        r = requests.get(url, timeout=15)

        if r.status_code == 200:
            data = r.json()
            structures = data.get(uid, [])

            for s in structures:
                pdb_data.append({
                    'uniprot_id': uid,
                    'pdb_id': s.get('pdb_id'),
                    'chain_id': s.get('chain_id'),
                    'resolution': s.get('resolution'),
                    'coverage': s.get('coverage'),
                    'method': s.get('experimental_method'),
                    'unp_start': s.get('unp_start'),
                    'unp_end': s.get('unp_end')
                })

        if i % 100 == 0 and i > 0:
            print(f" Progress: {i}/{len(unique_uids)} proteins checked. Mappings found: {len(pdb_data)}")

        time.sleep(0.1)

    except Exception as e:
        print(f" Skipping {uid} due to error: {e}")
        continue


df_pdb_info = pd.DataFrame(pdb_data)

df_pdb_info.to_csv("human_protein_pdb_structures.csv", index=False)

print("\n--- Processing Complete ---")
print(f"Bioactivity proteins processed: {len(unique_uids)}")
print(f"Total PDB entries found: {len(df_pdb_info)}")
print("Data saved to 'human_protein_pdb_structures.csv'")

df_pdb_info.head()

C:\Users\Anna_Maksymchuk1\AppData\Local\Temp\ipykernel_18612\3106396491.py:8: DtypeWarning: Columns (12,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df_results = pd.read_csv("human_swiss_prot_activities.csv")


Starting PDB fetch for 5327 unique proteins...
 Progress: 100/5327 proteins checked. Mappings found: 6926
 Progress: 200/5327 proteins checked. Mappings found: 11022
 Progress: 300/5327 proteins checked. Mappings found: 16822
 Progress: 400/5327 proteins checked. Mappings found: 20673
 Progress: 500/5327 proteins checked. Mappings found: 27728
 Progress: 600/5327 proteins checked. Mappings found: 32584
 Progress: 700/5327 proteins checked. Mappings found: 36163
 Progress: 800/5327 proteins checked. Mappings found: 41583
 Progress: 900/5327 proteins checked. Mappings found: 46968
 Progress: 1000/5327 proteins checked. Mappings found: 53418
 Progress: 1100/5327 proteins checked. Mappings found: 56406
 Progress: 1200/5327 proteins checked. Mappings found: 59316
 Progress: 1300/5327 proteins checked. Mappings found: 61680
 Progress: 1400/5327 proteins checked. Mappings found: 66140
 Progress: 1500/5327 proteins checked. Mappings found: 68062
 Progress: 1600/5327 proteins checked. Mappings 

,uniprot_id,pdb_id,chain_id,resolution,coverage,method,unp_start,unp_end
0,P36873,1jk7,A,1.90,1.0,X-ray diffraction,1,323
1,P36873,4ut2,A,1.96,1.0,X-ray diffraction,1,323
2,P36873,4ut2,B,1.96,1.0,X-ray diffraction,1,323
3,P36873,1it6,A,2.00,1.0,X-ray diffraction,1,323
4,P36873,1it6,B,2.00,1.0,X-ray diffraction,1,323


**Textual Data Extraction (NCBI PubMed API)**

This step uses the unique identifiers (`pubmed_id`) obtained from ChEMBL to fetch the original research abstracts from the National Center for Biotechnology Information (NCBI).

Dataset 4: PubMed Abstracts
* Source: NCBI Entrez Programming Utilities (E-utils).
* Purpose: Provides the primary textual corpus for the information retrieval system.
* Key Attributes Extracted:
    * `pubmed_id`: The primary key used to link articles to drugs and proteins in the warehouse.
    * `abstract`: The full text of the scientific summary, which will be preprocessed for search.
    
Retrieving data from NCBI requires adherence to their usage policies to avoid IP blocking.
Key Technical Implementations:
* Batching Strategy: The code processes IDs in groups of 200. This reduces the number of server requests and ensures the XML response size is manageable.
* NCBI Entrez Protocol: Uses efetch to retrieve metadata in XML format, which is then parsed using Biopython's `Entrez.read` to navigate the scientific schema.
* Rate Limiting: A time.sleep(0.5) delay is implemented between batches.
* Error Handling (Missing Data): Scientific papers like "Notes" or "Editorials" often lack a formal abstract. The try-except block handles KeyError to ensure the pipeline doesn't crash when an abstract is missing.

In [ ]:
Entrez.email = "anna.i.maksymchuk@gmail.com"  # Required by NCBI


df_activities = pd.read_csv("human_swiss_prot_activities.csv")
pmids = df_activities['pubmed_id'].dropna().unique().astype(int).astype(str).tolist()

abstract_data = []

print(f"Fetching abstracts for {len(pmids)} articles...")

batch_size = 200
for i in range(0, len(pmids), batch_size):
    batch = pmids[i:i+batch_size]
    try:
        handle = Entrez.efetch(db="pubmed", id=",".join(batch), rettype="abstract", retmode="xml")
        records = Entrez.read(handle)

        for article in records['PubmedArticle']:
            pmid = str(article['MedlineCitation']['PMID'])
            try:

                abstract_text = "".join(article['MedlineCitation']['Article']['Abstract']['AbstractText'])
                abstract_data.append({"pubmed_id": pmid, "abstract": abstract_text})
            except KeyError:

                abstract_data.append({"pubmed_id": pmid, "abstract": ""})

        print(f" Progress: {i + len(batch)}/{len(pmids)} abstracts retrieved.")
        time.sleep(0.5)

    except Exception as e:
        print(f" Error fetching batch starting at {i}: {e}")

df_articles = pd.DataFrame(abstract_data)
df_articles.to_csv("article_abstracts.csv", index=False)
print("SUCCESS: Abstracts saved to 'article_abstracts.csv'")

C:\Users\Anna_Maksymchuk1\AppData\Local\Temp\ipykernel_18612\533414506.py:9: DtypeWarning: Columns (12,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df_activities = pd.read_csv("human_swiss_prot_activities.csv")


Fetching abstracts for 36351 articles...
 Progress: 200/36351 abstracts retrieved.
 Progress: 400/36351 abstracts retrieved.
 Progress: 600/36351 abstracts retrieved.
 Progress: 800/36351 abstracts retrieved.
 Progress: 1000/36351 abstracts retrieved.
 Progress: 1200/36351 abstracts retrieved.
 Progress: 1400/36351 abstracts retrieved.
 Progress: 1600/36351 abstracts retrieved.
 Progress: 1800/36351 abstracts retrieved.
 Progress: 2000/36351 abstracts retrieved.
 Progress: 2200/36351 abstracts retrieved.
 Progress: 2400/36351 abstracts retrieved.
 Progress: 2600/36351 abstracts retrieved.
 Progress: 2800/36351 abstracts retrieved.
 Progress: 3000/36351 abstracts retrieved.
 Progress: 3200/36351 abstracts retrieved.
 Progress: 3400/36351 abstracts retrieved.
 Progress: 3600/36351 abstracts retrieved.
 Progress: 3800/36351 abstracts retrieved.
 Progress: 4000/36351 abstracts retrieved.
 Progress: 4200/36351 abstracts retrieved.
 Progress: 4400/36351 abstracts retrieved.
 Progress: 4600/3

In [ ]:
df_articles.head()

,pubmed_id,abstract
0,14695814,Okadaic acid (OA) is a toxin responsible for d...
1,14695824,A variety of novel heterocyclic compounds havi...
2,14695825,The hemoglobin-degrading aspartic proteases pl...
3,14695828,Recently we reported the pharmacological chara...
4,14695833,A series of benzoxazinones has been synthesize...


**Star schema**

A Dimensional Model optimized for searching drug-target interactions.

**Grain Selection**

The grain of our fact table is defined as one unique bioactivity measurement per experiment. This allows users to search for specific potencies (IC50) while filtering by protein properties or publication dates.

Table descriptions:
1. FactBioactivity: Stores numerical metrics like potency and confidence scores.
2. DimProtein: Contains biological identity data from UniProt.
3. DimDrug: Contains chemical identity data from ChEMBL.
4. DimArticle: The primary source for the information retrieval system, containing the abstract text and metadata. DimStructure: Maps proteins to their high-quality 3D structures from PDBe.
5. DimStructure: Maps proteins to their high-quality 3D structures from PDBe.



## Schema Feasibility Analysis — What Can We Extract From Each Source?

The table below maps every proposed column to its data source, the exact field
name in the API/DB response, and whether it is **directly available ✅**,
**derivable with transformation 🔧**, or **not available / requires
alternative strategy ❌**.

---

### FactBioactivity

| Column | Source | Available? | Notes |
|---|---|---|---|
| `activity_id` | ChEMBL `activities.activity_id` | ✅ | Used as natural PK |
| `protein_key` (FK) | Surrogate key we assign | ✅ | Derived from `uniprot_id` lookup |
| `drug_key` (FK) | Surrogate key we assign | ✅ | Derived from `drug_chembl_id` lookup |
| `article_key` (FK) | Surrogate key we assign | ✅ | Derived from `pubmed_id` lookup |
| `standard_type` | ChEMBL `activities.standard_type` | ✅ | IC50, Ki, Kd, etc. – already in the query |
| `standard_value` | ChEMBL `activities.standard_value` | ✅ | Filtered `IS NOT NULL` in the query |
| `standard_units` | ChEMBL `activities.standard_units` | ✅ | nM, µM, etc. |
| `pchembl_value` | ChEMBL `activities.pchembl_value` | ✅ | −log₁₀ of molar potency, already extracted |
| `confidence_score` | ChEMBL `assays.confidence_score` | ✅ | 0–9 quality score, already in the query |
| `assay_type` | ChEMBL `assays.assay_type` | ✅ | B/F/A/T/U codes – already in the query |
| `assay_description` | ChEMBL `assays.description` | ✅ | Free-text assay description, already extracted as `assay_description` |

> **All FactBioactivity columns are already in the ChEMBL SQL query** (`_BIOACTIVITY_QUERY`
> in `chembl_extractor.py`).  `assay_type` and `assay_description` **are** provided by ChEMBL –
> no information extraction needed.

---

### DimProtein

| Column | Source | Available? | Notes |
|---|---|---|---|
| `protein_key` | Assigned by pipeline | ✅ | Row-number surrogate key |
| `uniprot_id` | UniProt `Entry` | ✅ | Already extracted as `accession` |
| `protein_name` | UniProt `Protein names` | ✅ | Already in `_DEFAULT_FIELDS` |
| `gene_names` | UniProt `Gene Names` | ✅ | Already in `_DEFAULT_FIELDS` |
| `organism` | UniProt field `organism_name` | 🔧 | Add `organism_name` to `fields` param in `UniProtExtractor` |
| `protein_sequence` | UniProt field `sequence` | 🔧 | Add `sequence` to `fields` param — large column (~1 KB/protein) |
| `protein_class` | UniProt field `protein_families` | 🔧 | Add `protein_families` to fields; maps roughly to Kinase/GPCR/etc. |
| `ec_number` | UniProt field `ec` | 🔧 | Add `ec` to fields; blank for non-enzymes |
| `catalyzed_reaction` | UniProt field `cc_catalytic_activity` | 🔧 | Add `cc_catalytic_activity` to fields; blank for non-enzymes |

> **Action required:** Extend `_DEFAULT_FIELDS` in `uniprot_extractor.py` with
> `organism_name,sequence,protein_families,ec,cc_catalytic_activity`.

---

### DimDrug

| Column | Source | Available? | Notes |
|---|---|---|---|
| `drug_key` | Assigned by pipeline | ✅ | Row-number surrogate key |
| `drug_chembl_id` | ChEMBL `molecule_dictionary.chembl_id` | ✅ | Already extracted |
| `drug_name` | ChEMBL `molecule_dictionary.pref_name` | ✅ | Already extracted as `drug_name` |
| `molecule_type` | ChEMBL `molecule_dictionary.molecule_type` | 🔧 | Add `mol.molecule_type` to `_BIOACTIVITY_QUERY` |
| `molecular_weight` | ChEMBL `compound_properties.mw_freebase` | 🔧 | Add a JOIN to `compound_properties` table in the SQL query |
| `canonical_smiles` | ChEMBL `compound_structures.canonical_smiles` | 🔧 | Add a JOIN to `compound_structures` table in the SQL query |

> **Action required:** Extend `_BIOACTIVITY_QUERY` in `chembl_extractor.py` with two
> additional JOINs and three extra SELECT columns.

---

### DimArticle

| Column | Source | Available? | Notes |
|---|---|---|---|
| `article_key` | Assigned by pipeline | ✅ | Row-number surrogate key |
| `pubmed_id` | ChEMBL `docs.pubmed_id` | ✅ | Already extracted |
| `article_title` | ChEMBL `docs.title` | ✅ | Already extracted as `article_title` |
| `journal` | ChEMBL `docs.journal` | ✅ | Already extracted |
| `year` | ChEMBL `docs.year` | ✅ | Already extracted |
| `abstract` | PubMed `efetch` XML | ✅ | Already extracted by `PubMedExtractor` |
| `doi` | ChEMBL `docs.doi` | 🔧 | Add `doc.doi` to `_BIOACTIVITY_QUERY` SELECT clause |
| `first_author` | PubMed `efetch` XML | 🔧 | Extend `_parse_batch` in `PubMedExtractor` to also parse `AuthorList` |
| `authors` (full list) | PubMed XML `AuthorList` | 🔧 | Parse `LastName + ForeName` from each author node in Entrez response |

> **Sub-dimension Date:** `year` is already extracted.  Full `publication_date`
> (day + month) and `month` are available in the PubMed XML under
> `PubDate` — extend `PubMedExtractor` to parse them.  Alternatively, keep
> them as extra columns on `DimArticle` rather than a separate table.

---

### DimStructure

| Column | Source | Available? | Notes |
|---|---|---|---|
| `structure_key` | Assigned by pipeline | ✅ | Row-number surrogate key |
| `pdb_id` | PDBe `pdb_id` | ✅ | Already extracted |
| `chain_id` | PDBe `chain_id` | ✅ | Already extracted |
| `uniprot_id` | PDBe (join key) | ✅ | Already stored alongside each structure row |
| `resolution` | PDBe `resolution` | ✅ | Already extracted |
| `coverage` | PDBe `coverage` | ✅ | Already extracted |
| `method` | PDBe `experimental_method` | ✅ | Already extracted (renamed to `method`) |
| `unp_start` | PDBe `unp_start` | ✅ | Already extracted |
| `unp_end` | PDBe `unp_end` | ✅ | Already extracted |
| `resolution_units` | Constant — always Ångström | 🔧 | Hardcode `"Å"` during transformation; not in the API response |

---

### DimMetadata (proposed)

| Column | Source | Available? | Notes |
|---|---|---|---|
| Measure descriptions (IC50, Ki …) | ❌ No API provides these | 🔧 | Best approach: a **static lookup table** maintained manually or seeded from a curated list (e.g., the [ChEMBL measurement descriptions](https://www.ebi.ac.uk/chembl/explore/activities)) |
| Method descriptions (X-ray, NMR…) | ❌ No API | 🔧 | Same: small static lookup table keyed on `experimental_method` string |

> A `DimMetadata` table is feasible but must be populated from a **static CSV file
> you curate once**, not from an API.  It contains ~10–20 rows at most.  The
> simplest approach is to store it in `data/metadata/` and load it during the
> transformation stage like any other dimension.

---

### Summary of Required Code Changes

| Extractor / File | Change needed |
|---|---|
| `uniprot_extractor.py` | Add 5 fields to `_DEFAULT_FIELDS`: `organism_name,sequence,protein_families,ec,cc_catalytic_activity` |
| `chembl_extractor.py` | Add `mol.molecule_type` to SELECT; add JOINs to `compound_properties` and `compound_structures`; add `doc.doi` |
| `pubmed_extractor.py` | Parse `doi`, `authors` (LastName + ForeName list), `pub_date` from Entrez XML |
| `transformation/cleaner.py` | Handle new nullable columns; `resolution_units` constant = `"Å"` |
| `transformation/dimensional_builder.py` | Add new columns to each `build_dim_*` method |
| `database/schema.sql` | Add new columns to the relevant `CREATE TABLE` statements |
| `database/schema.py` (TABLE_COLUMNS) | Add new column names to each table list |
| `data/metadata/` *(new)* | Seed CSV for `DimMetadata` — curated manually |